# Task 1: Web Scraping & Data Exploration
## Fintech Review Analytics

**Objective**: Collect 1,200+ reviews (400+ per bank) from Google Play Store, preprocess the data, and prepare for sentiment analysis.

**Target KPIs**:
- ✅ 1,200+ reviews collected
- ✅ <5% missing data rate
- ✅ 5-column clean CSV (review, rating, date, bank, source)
- ✅ CI/CD workflow passes
- ✅ Conventional commit messages

In [13]:
!pip install google-play-scraper

## Section 1: Setup & Configuration

Import required libraries, configure logging, and set up data directories.

In [14]:
import sys
import os
import logging
from datetime import datetime
from pathlib import Path
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google_play_scraper import reviews_all
from tqdm import tqdm

warnings.filterwarnings('ignore')

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Set up paths
PROJECT_ROOT = Path('../')
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'

# Create directories if they don't exist
DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# Add src to path
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

# Configure plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

logger.info("Setup complete. Ready for scraping...")

2026-05-17 12:27:25,131 - __main__ - INFO - Setup complete. Ready for scraping...


## Section 2: Web Scraping from Google Play Store

Configure bank apps and scrape reviews. Target: 400+ reviews per bank (1,200 total)

In [16]:
        # Scrape reviews using google-play-scraper
        reviews_data = reviews_all(
            package_id,
            sleep_milliseconds=100,
            lang='en',
            country='us'
        )

## Section 3: Data Preprocessing & Cleaning

Clean the raw data: remove duplicates, handle missing values, normalize dates, validate columns.

In [17]:
# Load the raw data
if df_raw.empty:
    print("⚠️ ERROR: No data to preprocess. The raw dataframe is empty.")
    print("This likely means the scraping failed. Check the scraping section above.")
    df_clean = pd.DataFrame(columns=['review', 'rating', 'date', 'bank', 'source'])
else:
    df = df_raw.copy()
    
    logger.info(f"Starting with {len(df)} raw reviews")
    
    # Step 1: Check for duplicates
    logger.info("\n--- Step 1: Remove Duplicates ---")
    initial_count = len(df)
    df_dedup = df.drop_duplicates(
        subset=['review', 'bank', 'date'],
        keep='first'
    )
    duplicates_removed = initial_count - len(df_dedup)
    logger.info(f"Duplicates removed: {duplicates_removed}")
    df = df_dedup
    
    # Step 2: Handle missing values
    logger.info("\n--- Step 2: Handle Missing Values ---")
    initial_count = len(df)
    
    # Check missing values
    missing_counts = df[['review', 'rating', 'date', 'bank', 'source']].isnull().sum()
    logger.info("Missing values before cleaning:")
    for col, count in missing_counts.items():
        if count > 0:
            pct = (count / len(df) * 100)
            logger.info(f"  {col}: {count} ({pct:.2f}%)")
    
    # Drop rows with missing critical fields
    df = df.dropna(subset=['review', 'rating'])
    rows_dropped = initial_count - len(df)
    logger.info(f"Rows dropped (missing critical fields): {rows_dropped}")
    
    # Fill non-critical missing values
    df['bank'] = df['bank'].fillna('Unknown')
    df['source'] = df['source'].fillna('Unknown')
    
    # Step 3: Normalize dates
    logger.info("\n--- Step 3: Normalize Dates ---")
    df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')
    logger.info(f"Dates normalized to YYYY-MM-DD format")
    
    # Step 4: Validate ratings
    logger.info("\n--- Step 4: Validate Ratings ---")
    initial_count = len(df)
    df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
    df = df[(df['rating'] >= 1) & (df['rating'] <= 5)]
    invalid_removed = initial_count - len(df)
    if invalid_removed > 0:
        logger.info(f"Invalid ratings removed: {invalid_removed}")
    
    # Step 5: Select required columns
    logger.info("\n--- Step 5: Select Required Columns ---")
    REQUIRED_COLUMNS = ['review', 'rating', 'date', 'bank', 'source']
    df_clean = df[REQUIRED_COLUMNS].copy()
    logger.info(f"Selected columns: {REQUIRED_COLUMNS}")
    
    # Calculate metrics
    logger.info("\n" + "="*60)
    logger.info("PREPROCESSING SUMMARY")
    logger.info("="*60)
    logger.info(f"Initial records: {initial_count}")
    logger.info(f"Final records: {len(df_clean)}")
    logger.info(f"Duplicates removed: {duplicates_removed}")
    logger.info(f"Rows dropped (missing data): {rows_dropped + invalid_removed}")
    missing_pct = ((rows_dropped + invalid_removed) / initial_count * 100) if initial_count > 0 else 0
    logger.info(f"Missing data percentage: {missing_pct:.2f}%")
    logger.info(f"✓ Target: <5% missing data - {'PASS' if missing_pct < 5 else 'FAIL'}")
    
    # Save cleaned data
    cleaned_csv_path = DATA_PROCESSED / 'reviews_cleaned.csv'
    df_clean.to_csv(cleaned_csv_path, index=False)
    logger.info(f"\nCleaned data saved to: {cleaned_csv_path}")
    
    print(f"\n✅ Preprocessing complete!")
    print(f"   - Input:  {len(df_raw)} reviews")
    print(f"   - Output: {len(df_clean)} reviews")
    print(f"   - Quality: {100 - missing_pct:.1f}% (target: >95%)")

⚠️ ERROR: No data to preprocess. The raw dataframe is empty.
This likely means the scraping failed. Check the scraping section above.


## Section 4: Exploratory Data Analysis

Explore the cleaned dataset: distributions, statistics, quality checks.

In [18]:
if df_clean.empty:
    print("⚠️ No data available for analysis. Skipping EDA.")
else:
    # Display basic statistics
    print("Dataset Overview:")
    print(f"  Total reviews: {len(df_clean)}")
    print(f"  Date range: {df_clean['date'].min()} to {df_clean['date'].max()}")
    print(f"  Banks: {df_clean['bank'].nunique()}")
    print()
    
    # Distribution by bank
    print("Reviews by Bank:")
    bank_counts = df_clean['bank'].value_counts()
    for bank, count in bank_counts.items():
        pct = (count / len(df_clean) * 100)
        print(f"  {bank}: {count} ({pct:.1f}%)")
    print()
    
    # Rating distribution
    print("Rating Distribution:")
    rating_counts = df_clean['rating'].value_counts().sort_index()
    for rating, count in rating_counts.items():
        pct = (count / len(df_clean) * 100)
        bar = '█' * int(pct / 2)
        print(f"  {int(rating)} stars: {count:4d} ({pct:5.1f}%) {bar}")
    print()
    
    # Summary statistics
    print("Summary Statistics:")
    print(df_clean.describe())

⚠️ No data available for analysis. Skipping EDA.


In [20]:
if df_clean.empty:
    print("⚠️ No data available for visualizations. Skipping plots.")
else:
    # Visualizations
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Calculate the stats we need
    bank_counts = df_clean['bank'].value_counts()
    rating_counts = df_clean['rating'].value_counts().sort_index()
    
    # 1. Reviews by Bank
    ax1 = axes[0, 0]
    bank_counts.plot(kind='bar', ax=ax1, color='skyblue')
    ax1.set_title('Reviews per Bank', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Bank')
    ax1.set_ylabel('Count')
    ax1.tick_params(axis='x', rotation=45)
    
    # 2. Rating Distribution
    ax2 = axes[0, 1]
    rating_counts.plot(kind='bar', ax=ax2, color='coral')
    ax2.set_title('Rating Distribution', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Star Rating')
    ax2.set_ylabel('Count')
    ax2.tick_params(axis='x', rotation=0)
    
    # 3. Average rating by bank
    ax3 = axes[1, 0]
    avg_rating = df_clean.groupby('bank')['rating'].mean()
    avg_rating.plot(kind='barh', ax=ax3, color='lightgreen')
    ax3.set_title('Average Rating by Bank', fontsize=12, fontweight='bold')
    ax3.set_xlabel('Average Rating')
    ax3.set_ylabel('Bank')
    
    # 4. Reviews per day (time series)
    ax4 = axes[1, 1]
    df_clean['date'] = pd.to_datetime(df_clean['date'])
    reviews_per_day = df_clean.groupby('date').size()
    ax4.plot(reviews_per_day.index, reviews_per_day.values, marker='o', linestyle='-', linewidth=1)
    ax4.set_title('Reviews Over Time', fontsize=12, fontweight='bold')
    ax4.set_xlabel('Date')
    ax4.set_ylabel('Number of Reviews')
    ax4.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    print("✓ Visualizations generated")

⚠️ No data available for visualizations. Skipping plots.


## Section 5: KPI Validation

Verify that all Key Performance Indicators are met for Task 1.

In [22]:
# KPI Validation
print("="*60)
print("TASK 1: KEY PERFORMANCE INDICATORS (KPIs)")
print("="*60)

# KPI 1: 1,200+ reviews collected
kpi1_target = 1200
kpi1_actual = len(df_clean)
kpi1_pass = kpi1_actual >= kpi1_target
print(f"\n✓ KPI 1: Reviews Collected")
print(f"  Target: {kpi1_target}+")
print(f"  Actual: {kpi1_actual}")
print(f"  Status: {'✅ PASS' if kpi1_pass else '❌ FAIL'}")

# KPI 2: <5% missing data
initial_records = len(df_raw)
missing_records = initial_records - len(df_clean)
missing_pct = (missing_records / initial_records * 100) if initial_records > 0 else 0
kpi2_target = 5
kpi2_pass = missing_pct < kpi2_target
print(f"\n✓ KPI 2: Missing Data Rate")
print(f"  Target: <{kpi2_target}%")
print(f"  Actual: {missing_pct:.2f}%")
print(f"  Status: {'✅ PASS' if kpi2_pass else '❌ FAIL'}")

# KPI 3: Required columns present
kpi3_columns = ['review', 'rating', 'date', 'bank', 'source']
kpi3_pass = all(col in df_clean.columns for col in kpi3_columns)
print(f"\n✓ KPI 3: Required Columns")
print(f"  Required: {kpi3_columns}")
print(f"  Present: {list(df_clean.columns)}")
print(f"  Status: {'✅ PASS' if kpi3_pass else '❌ FAIL'}")

# KPI 4: Reviews per bank
print(f"\n✓ KPI 4: Reviews per Bank (min. 400)")
if not df_clean.empty:
    bank_counts = df_clean['bank'].value_counts()
    for bank in bank_counts.index:
        count = bank_counts[bank]
        status = '✅' if count >= 400 else '❌'
        print(f"  {status} {bank}: {count}")
    kpi4_pass = (bank_counts >= 400).all()
else:
    print("  ⚠️ No data available")
    kpi4_pass = False

# Summary
print(f"\n{'='*60}")
all_pass = kpi1_pass and kpi2_pass and kpi3_pass and kpi4_pass
print(f"Overall Status: {'✅ ALL KPIs MET' if all_pass else '⚠️ Some KPIs not met'}")
print(f"{'='*60}")

TASK 1: KEY PERFORMANCE INDICATORS (KPIs)

✓ KPI 1: Reviews Collected
  Target: 1200+
  Actual: 0
  Status: ❌ FAIL

✓ KPI 2: Missing Data Rate
  Target: <5%
  Actual: 0.00%
  Status: ✅ PASS

✓ KPI 3: Required Columns
  Required: ['review', 'rating', 'date', 'bank', 'source']
  Present: ['review', 'rating', 'date', 'bank', 'source']
  Status: ✅ PASS

✓ KPI 4: Reviews per Bank (min. 400)
  ⚠️ No data available

Overall Status: ⚠️ Some KPIs not met
